# Candelaria Vial: priorización preventiva con datos abiertos

**Asignatura:** Técnicas de Aprendizaje de Máquina  
**Integrantes:** Completar nombres y códigos  
**Fuente principal:** [Accidentalidad Vial Municipio de Candelaria, Valle](https://www.datos.gov.co/Transporte/Accidentalidad-Vial-Municipio-de-Candelaria-Valle-/7wbf-88zm)

> **Nota de reproducibilidad.** Ejecutar el cuaderno de arriba hacia abajo y anotar la fecha de descarga: el conjunto abierto puede actualizarse.

## 1. Elevator pitch

La Secretaría de Tránsito y Transporte de Candelaria debe decidir dónde focalizar controles, señalización, inspecciones y educación vial. Decidir por intuición o por el accidente más reciente puede dispersar recursos escasos. Nuestra propuesta usa los registros históricos de siniestros para anticipar el comportamiento del **próximo mes por corregimiento** mediante dos preguntas complementarias.

**Stakeholder:** Secretaría de Tránsito y Transporte de Candelaria, Valle del Cauca.  
**Costo de inacción:** intervenciones tardías o mal focalizadas; costos humanos, de atención y de movilidad.  
**Criterio de éxito:** superar referencias simples en una prueba temporal y mantener una interpretación preventiva y agregada de los resultados.

### Problemas

1. **Regresión:** ¿cuántos siniestros se esperan el próximo mes por corregimiento?
2. **Clasificación:** ¿el próximo mes tendrá alta siniestralidad un corregimiento?

### Hipótesis

1. La frecuencia mensual de siniestros no es homogénea entre corregimientos.
2. Los meses con mayor actividad reciente tienen mayor probabilidad de entrar en un escenario de alta siniestralidad durante el siguiente periodo.
3. Los siniestros recientes, junto con calendario y corregimiento, predicen mejor el volumen mensual que una regla plana basada en el promedio histórico.

## 2. IA como revisora crítica: definición del problema

**Prompt usado:**

> Actúa como directora de movilidad de Candelaria. Queremos priorizar intervenciones de seguridad vial con registros históricos y responder dos preguntas por corregimiento-mes: cuántos siniestros se esperan el próximo mes y si ese mes tendrá alta siniestralidad. Dame razones concretas para no financiar el proyecto, qué evidencia pedirías y qué daño podría causar un falso positivo o falso negativo.

**Respuesta crítica sintetizada:** los registros pueden tener subregistro y cambios administrativos; una mayor frecuencia no prueba causalidad; una alerta puede desplazar recursos hacia zonas ya más vigiladas; y sin evaluación temporal un modelo puede parecer útil por simple inercia histórica.

**Acciones del equipo:** se separa el último año para prueba, se agregan meses con cero siniestros al panel, se usan únicamente variables disponibles antes del mes objetivo y la app se define como apoyo a inspección, nunca como mecanismo sancionatorio.

In [ ]:
# Instalación para Google Colab (ejecutar solo una vez)
!pip -q install pandas numpy scikit-learn seaborn plotly

import warnings
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import (
    accuracy_score, average_precision_score, confusion_matrix, f1_score,
    mean_absolute_error, mean_squared_error, precision_score, r2_score,
    recall_score, roc_auc_score, RocCurveDisplay, PrecisionRecallDisplay,
)
from sklearn.model_selection import ParameterSampler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier, DecisionTreeRegressor

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", palette="deep")
RANDOM_STATE = 42
DATA_URL = "https://www.datos.gov.co/resource/7wbf-88zm.csv?$limit=100000"
DOWNLOAD_TIME = datetime.now(timezone.utc).isoformat()

In [ ]:
# Descarga reproducible desde la API Socrata
raw = pd.read_csv(DATA_URL)
expected_columns = {
    "clase_de_accidente", "gravedad", "vias", "corregimiento",
    "d_a_semana", "fecha_de_ocurrecia", "hora_ocurrencia",
}
assert expected_columns.issubset(raw.columns), "El esquema publicado cambió; revisar el diccionario."
print(f"Fecha UTC de descarga: {DOWNLOAD_TIME}")
print(f"Filas: {len(raw):,}; columnas: {raw.shape[1]}")
raw.head()

## 3. Diccionario y calidad de los datos

| Campo | Tipo publicado | Uso y justificación |
| --- | --- | --- |
| `clase_de_accidente` | Texto/categórica | Variable descriptiva para EDA. Se excluye de los predictores preventivos porque se conoce con el evento. |
| `gravedad` | Texto/categórica | Variable descriptiva del EDA. Se conserva para estudiar composición y posibles cambios de registro, pero ya no es el objetivo de clasificación. |
| `vias` | Texto/categórica | Contexto espacial utilizado en el EDA; no entra en los modelos mensuales de esta versión. |
| `corregimiento` | Texto/categórica | Unidad territorial directamente accionable para la priorización mensual. |
| `d_a_semana` | Texto/categórica | Se contrasta con la fecha para revisar consistencia y patrones temporales en el EDA. |
| `fecha_de_ocurrecia` | Texto/fecha | Fuente para año, mes y periodo temporal; permite la partición cronológica. |
| `hora_ocurrencia` | Texto/hora | Se transforma a hora numérica para el EDA de patrones horarios. |
| `anio`, `mes`, `dia_semana_num`, `hora` | Numéricas derivadas | Variables de calendario obtenidas exclusivamente de fecha/hora. |
| `es_fin_de_semana` | Binaria derivada | Variable temporal descriptiva para el EDA. |
| `mes_seno`, `mes_coseno`, `hora_seno`, `hora_coseno` | Numéricas derivadas | Representan la naturaleza cíclica de meses y horas. |
| `siniestros_lag_1` | Numérica derivada | Número de siniestros del mes inmediatamente anterior en el mismo corregimiento; solo usa información pasada. |
| `promedio_3_meses_previo` | Numérica derivada | Promedio de los tres meses anteriores; resume tendencia reciente sin mirar el futuro. |
| `siniestros_mes` | Numérica objetivo | Conteo mensual de siniestros por corregimiento para el problema de regresión. |
| `alta_siniestralidad` | Binaria objetivo | Objetivo de clasificación, construido después de la partición temporal con un umbral calculado solo sobre entrenamiento. |
| `lesion_o_muerte` | Binaria derivada | Se conserva únicamente como apoyo descriptivo del EDA de severidad; no es objetivo ni predictor del modelado final. |

**Datos complementarios.** En esta versión no se incorporan fuentes externas: el objetivo es evaluar primero cuánto valor aporta el conjunto oficial por sí solo. Como trabajo futuro se propone integrar aforos/exposición vial, clima, obras, controles y geometría vial mediante llaves temporales y espaciales verificadas. La fuente es un registro administrativo; por ello, un cero o una frecuencia baja puede reflejar subregistro, no necesariamente ausencia absoluta de siniestros.

In [ ]:
quality = pd.DataFrame({
    "tipo": raw.dtypes.astype(str),
    "nulos": raw.isna().sum(),
    "n_unicos": raw.nunique(dropna=True),
})
display(quality)
display(raw.duplicated().value_counts().rename("filas"))

In [ ]:
def clean_text(series):
    return (series.fillna("SIN_DATO").astype(str).str.strip().str.upper()
            .replace({"": "SIN_DATO", "NAN": "SIN_DATO"}))

incidents = raw.copy()
for column in ["vias", "corregimiento", "d_a_semana", "gravedad", "clase_de_accidente"]:
    incidents[column] = clean_text(incidents[column])
incidents["fecha"] = pd.to_datetime(incidents["fecha_de_ocurrecia"], dayfirst=True, errors="coerce")
parsed_time = pd.to_timedelta(incidents["hora_ocurrencia"], errors="coerce")
incidents["hora"] = (parsed_time.dt.total_seconds() / 3600).fillna(12).clip(0, 23.99)
incidents = incidents.dropna(subset=["fecha"]).copy()
incidents["anio"] = incidents.fecha.dt.year.astype(int)
incidents["mes"] = incidents.fecha.dt.month.astype(int)
incidents["dia_semana_num"] = incidents.fecha.dt.dayofweek.astype(int)
incidents["es_fin_de_semana"] = (incidents.dia_semana_num >= 5).astype(int)
incidents["mes_seno"] = np.sin(2 * np.pi * incidents.mes / 12)
incidents["mes_coseno"] = np.cos(2 * np.pi * incidents.mes / 12)
incidents["hora_seno"] = np.sin(2 * np.pi * incidents.hora / 24)
incidents["hora_coseno"] = np.cos(2 * np.pi * incidents.hora / 24)
incidents["lesion_o_muerte"] = incidents.gravedad.isin(["HERIDOS", "MUERTOS"]).astype(int)

display(incidents[["fecha", "hora", "gravedad", "vias", "corregimiento"]].head())
print("Cobertura:", incidents.fecha.min().date(), "a", incidents.fecha.max().date())

## 4. EDA con narrativa

Cada gráfica responde a una decisión de negocio: ¿la cobertura es comparable por año?, ¿dónde conviene revisar primero?, ¿qué periodos conviene considerar en el plan preventivo? No se deben convertir correlaciones o conteos en afirmaciones de causalidad.

### IA como revisora crítica — EDA

**Prompt usado:**

> Estoy analizando un conjunto de accidentalidad vial para decidir dónde priorizar intervenciones. Mi EDA contiene conteos por año, mes, corregimiento y una tabla hora-día. ¿Qué análisis estadísticos mínimos faltan para evitar una narrativa puramente descriptiva y qué sesgos debo revisar?

**Crítica sintetizada:** faltaban medidas explícitas de tendencia central y dispersión, asimetría y correlaciones; además, los cambios interanuales no deben interpretarse como cambios reales de riesgo sin considerar cobertura y registro.

**Acción tomada:** se agregó una tabla con media, mediana, desviación estándar, IQR, mínimo, máximo y `skewness`, una matriz de correlaciones y una interpretación no causal de los patrones.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sns.countplot(data=incidents, x="anio", hue="gravedad", ax=axes[0])
axes[0].set_title("¿La composición de severidad es comparable entre años?")
axes[0].set_xlabel("Año del siniestro")
sns.countplot(data=incidents, x="mes", hue="gravedad", ax=axes[1])
axes[1].set_title("¿Qué meses concentran los registros por gravedad?")
axes[1].set_xlabel("Mes")
plt.tight_layout()
plt.show()

top_locations = incidents.corregimiento.value_counts().head(10).sort_values()
ax = top_locations.plot.barh(figsize=(9, 5), title="¿Qué corregimientos requieren revisión de capacidad preventiva?")
ax.set_xlabel("Siniestros registrados")
plt.show()

In [ ]:
pivot = pd.crosstab(incidents["hora"].astype(int), incidents["dia_semana_num"], normalize="columns")
plt.figure(figsize=(10, 7))
sns.heatmap(pivot, cmap="YlOrRd", cbar_kws={"label": "Proporción dentro del día"})
plt.title("¿Qué franjas deben revisarse junto con los controles de campo?")
plt.xlabel("Día de la semana (0=Lunes)")
plt.ylabel("Hora")
plt.show()

severity_year = pd.crosstab(incidents.anio, incidents.gravedad, normalize="index").round(3)
display(severity_year)

**Hallazgo inesperado a discutir.** Si la proporción de `DAÑOS`, `HERIDOS` o `MUERTOS` cambia abruptamente entre años, no se debe atribuir de inmediato a una mejora o deterioro vial: puede representar un cambio de registro. Por eso la severidad se mantiene como parte del EDA y no como objetivo del problema de clasificación final.

In [ ]:
# Panel completo corregimiento-mes: evita entrenar solo con meses que tuvieron siniestros.
start, end = incidents.fecha.min().to_period("M"), incidents.fecha.max().to_period("M")
periods = pd.period_range(start, end, freq="M")
locations = pd.DataFrame({"corregimiento": sorted(incidents.corregimiento.unique())})
monthly = locations.merge(pd.DataFrame({"periodo": periods}), how="cross")
monthly["fecha"] = monthly.periodo.dt.to_timestamp()
observed = (incidents.assign(periodo=incidents.fecha.dt.to_period("M"))
            .groupby(["corregimiento", "periodo"], as_index=False).size()
            .rename(columns={"size": "siniestros_mes"}))
monthly = monthly.merge(observed, on=["corregimiento", "periodo"], how="left")
monthly["siniestros_mes"] = monthly.siniestros_mes.fillna(0).astype(int)
monthly = monthly.sort_values(["corregimiento", "fecha"]).reset_index(drop=True)
groups = monthly.groupby("corregimiento", observed=True).siniestros_mes
monthly["siniestros_lag_1"] = groups.shift(1).fillna(0)
monthly["promedio_3_meses_previo"] = groups.transform(lambda x: x.shift(1).rolling(3, min_periods=1).mean()).fillna(0)
monthly["anio"] = monthly.fecha.dt.year.astype(int)
monthly["mes"] = monthly.fecha.dt.month.astype(int)
monthly["mes_seno"] = np.sin(2 * np.pi * monthly.mes / 12)
monthly["mes_coseno"] = np.cos(2 * np.pi * monthly.mes / 12)
print(monthly.shape)
monthly.head()

In [ ]:
# Estadística descriptiva y sesgo: requisitos explícitos del EDA
# Pregunta: ¿qué tan disperso y asimétrico es el volumen de siniestros y qué variables numéricas
# podrían justificar transformaciones o cautela al interpretar relaciones?
eda_numeric = incidents[["anio", "mes", "dia_semana_num", "hora"]].describe().T
eda_numeric["mediana"] = incidents[["anio", "mes", "dia_semana_num", "hora"]].median()
eda_numeric["IQR"] = incidents[["anio", "mes", "dia_semana_num", "hora"]].quantile(0.75) - incidents[["anio", "mes", "dia_semana_num", "hora"]].quantile(0.25)
eda_numeric["sesgo_skew"] = incidents[["anio", "mes", "dia_semana_num", "hora"]].skew()
display(eda_numeric[["count", "mean", "std", "mediana", "IQR", "min", "max", "sesgo_skew"]].round(3))

# Correlaciones entre variables numéricas derivadas. Se interpretan como asociaciones, no causalidad.
corr_cols = ["mes", "dia_semana_num", "hora", "es_fin_de_semana", "mes_seno", "mes_coseno", "hora_seno", "hora_coseno", "lesion_o_muerte"]
corr = incidents[corr_cols].corr(numeric_only=True)
display(corr.round(3))
plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, fmt=".2f", center=0)
plt.title("Correlaciones entre variables temporales y severidad")
plt.tight_layout()
plt.show()

# Tendencia central y dispersión del conteo mensual por corregimiento.
monthly_summary = monthly.groupby("corregimiento")["siniestros_mes"].agg(
    media="mean", mediana="median", desviacion="std", minimo="min", maximo="max"
).sort_values("media", ascending=False)
display(monthly_summary.round(3))

## 5. Preparación y partición temporal

Usamos el último año disponible solo para prueba. Los lags se calculan con periodos anteriores y se utilizan en **ambos problemas mensuales**. `clase_de_accidente`, `gravedad` y cualquier consecuencia posterior al siniestro se conservan para el EDA, pero no se usan como predictores preventivos.

### IA como revisora crítica — preparación de datos

**Prompt usado:**

> Para dos modelos preventivos a nivel corregimiento-mes —regresión del conteo y clasificación de alta siniestralidad—, ¿qué riesgos de fuga de información debo revisar antes de entrenar? Evalúa especialmente la construcción del umbral y los conteos históricos.

**Crítica sintetizada:** el umbral de la clasificación no puede calcularse usando el año de prueba; los lags deben construirse solo con periodos anteriores; y el conteo del propio mes objetivo no puede entrar como predictor.

**Acción tomada:** los lags usan únicamente meses previos, el último año queda intacto como prueba y el umbral de alta siniestralidad se obtiene exclusivamente del periodo de entrenamiento.

In [ ]:
test_year = int(incidents.anio.max())
inc_train, inc_test = incidents[incidents.anio < test_year].copy(), incidents[incidents.anio == test_year].copy()
mon_train, mon_test = monthly[monthly.anio < test_year].copy(), monthly[monthly.anio == test_year].copy()

# Definición de alta siniestralidad sin mirar el año de prueba.
# Para que la abundancia de ceros no convierta 1 siniestro en "alta" automáticamente,
# el P75 se calcula solo entre meses con al menos un siniestro en entrenamiento.
positive_train = mon_train.loc[mon_train.siniestros_mes > 0, "siniestros_mes"]
p75_train = float(positive_train.quantile(0.75))
HIGH_THRESHOLD = max(1, int(np.floor(p75_train)) + 1)
mon_train["alta_siniestralidad"] = (mon_train.siniestros_mes >= HIGH_THRESHOLD).astype(int)
mon_test["alta_siniestralidad"] = (mon_test.siniestros_mes >= HIGH_THRESHOLD).astype(int)

print(f"Entrenamiento: hasta {test_year - 1}; prueba: {test_year}")
print("Incidentes train/test:", inc_train.shape, inc_test.shape)
print("Panel mensual train/test:", mon_train.shape, mon_test.shape)
print(f"P75 de meses activos en entrenamiento: {p75_train:.2f}")
print(f"Alta siniestralidad: {HIGH_THRESHOLD} o más siniestros en un mes")
print(mon_train.alta_siniestralidad.value_counts(normalize=True).rename("proporción"))

def preprocessor(categorical, numeric):
    return ColumnTransformer([
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                          ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=5))]), categorical),
        ("num", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scale", StandardScaler())]), numeric),
    ])

## 6. Problema 1 — regresión de siniestros mensuales

La salida es una cantidad esperada de siniestros en el corregimiento durante el mes. Se comparan **Regresión lineal, Árbol de decisión y Random Forest**, que corresponden a técnicas vistas en el curso. El promedio histórico se conserva únicamente como baseline de referencia. El año previo al test funciona como validación temporal para optimizar Árbol y Random Forest; Regresión lineal no tiene hiperparámetros equivalentes que requieran esa búsqueda.

### IA como revisora crítica — optimización e iteración

**Prompt usado:**

> Si un Random Forest supera al baseline en la prueba temporal, ¿basta con elegirlo? ¿Qué debería hacer para demostrar que la mejora no depende de hiperparámetros escogidos a mano y que no estoy usando el año de prueba para seleccionar el modelo?

**Crítica sintetizada:** no basta con comparar una configuración elegida manualmente; se necesita una validación temporal interna y dejar intacto el año de prueba.

**Acción tomada:** se añadió búsqueda de hiperparámetros sobre años anteriores al año de prueba, usando el año inmediatamente anterior como validación temporal, y se evalúan los parámetros elegidos una sola vez sobre el año de prueba.

In [ ]:
reg_cat = ["corregimiento"]
reg_num = ["anio", "mes", "mes_seno", "mes_coseno", "siniestros_lag_1", "promedio_3_meses_previo"]
reg_features = reg_cat + reg_num
reg_models = {
    "Regresión lineal": LinearRegression(),
    "Árbol de decisión": DecisionTreeRegressor(max_depth=6, min_samples_leaf=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=300, max_depth=10, min_samples_leaf=4,
                                             max_features=0.8, n_jobs=-1, random_state=RANDOM_STATE),
}

# Baseline simple, sin DummyRegressor.
baseline_pred = np.repeat(mon_train.siniestros_mes.mean(), len(mon_test))
reg_rows = [{
    "Modelo": "Baseline: promedio histórico",
    "MAE": mean_absolute_error(mon_test.siniestros_mes, baseline_pred),
    "RMSE": mean_squared_error(mon_test.siniestros_mes, baseline_pred) ** 0.5,
    "R2": r2_score(mon_test.siniestros_mes, baseline_pred),
}]

reg_fitted = {}
for name, estimator in reg_models.items():
    pipe = Pipeline([("prep", preprocessor(reg_cat, reg_num)), ("model", estimator)])
    pipe.fit(mon_train[reg_features], mon_train.siniestros_mes)
    pred = pipe.predict(mon_test[reg_features])
    reg_rows.append({"Modelo": name, "MAE": mean_absolute_error(mon_test.siniestros_mes, pred),
                     "RMSE": mean_squared_error(mon_test.siniestros_mes, pred) ** 0.5,
                     "R2": r2_score(mon_test.siniestros_mes, pred)})
    reg_fitted[name] = pipe
reg_results = pd.DataFrame(reg_rows).sort_values("MAE")
display(reg_results.round(4))

# Optimización temporal: el año previo al test funciona como validación.
reg_val_year = test_year - 1
reg_fit_tune = monthly[monthly.anio < reg_val_year].copy()
reg_val_tune = monthly[monthly.anio == reg_val_year].copy()
reg_search_spaces = {
    "Árbol de decisión": {
        "model__max_depth": [3, 4, 5, 6, 8, 10, None],
        "model__min_samples_leaf": [2, 4, 6, 8, 12, 16],
    },
    "Random Forest": {
        "model__n_estimators": [200, 300, 500],
        "model__max_depth": [5, 8, 10, 12, None],
        "model__min_samples_leaf": [2, 4, 6, 8],
        "model__max_features": [0.5, 0.8, 1.0],
    },
}

tuned_reg = {}
for name in ["Árbol de decisión", "Random Forest"]:
    sampled = list(ParameterSampler(reg_search_spaces[name], n_iter=10, random_state=RANDOM_STATE))
    best_mae, best_params = np.inf, None
    for params in sampled:
        candidate = Pipeline([("prep", preprocessor(reg_cat, reg_num)), ("model", reg_models[name])])
        candidate.set_params(**params)
        candidate.fit(reg_fit_tune[reg_features], reg_fit_tune.siniestros_mes)
        val_pred = candidate.predict(reg_val_tune[reg_features])
        mae = mean_absolute_error(reg_val_tune.siniestros_mes, val_pred)
        if mae < best_mae:
            best_mae, best_params = mae, params
    final_pipe = Pipeline([("prep", preprocessor(reg_cat, reg_num)), ("model", reg_models[name])])
    final_pipe.set_params(**best_params)
    final_pipe.fit(mon_train[reg_features], mon_train.siniestros_mes)
    tuned_reg[name] = final_pipe
    print(f"{name}: mejor validación temporal MAE={best_mae:.4f}; parámetros={best_params}")

tuned_rows = []
for name, pipe in tuned_reg.items():
    pred = pipe.predict(mon_test[reg_features])
    tuned_rows.append({"Modelo optimizado": name,
                       "MAE": mean_absolute_error(mon_test.siniestros_mes, pred),
                       "RMSE": mean_squared_error(mon_test.siniestros_mes, pred) ** 0.5,
                       "R2": r2_score(mon_test.siniestros_mes, pred)})
tuned_reg_results = pd.DataFrame(tuned_rows).sort_values("MAE")
display(tuned_reg_results.round(4))

**Interpretación requerida:** compare la mejora absoluta y relativa de MAE frente al baseline. Un R² bajo no debe ocultarse: señala que faltan variables relevantes, como flujo vehicular, clima, obras, controles y exposición de actores viales. La salida solo sirve para orientar una revisión humana del territorio.

## 7. Problema 2 — clasificación de alta siniestralidad

La pregunta es: **¿el próximo mes tendrá alta siniestralidad un corregimiento?** La unidad sigue siendo corregimiento-mes. La etiqueta vale 1 cuando el conteo mensual alcanza el umbral calculado solo con entrenamiento y 0 en caso contrario. Se comparan Regresión logística, Árbol de decisión y Random Forest. Se reportan Accuracy, Precision, Recall, F1, AUC-ROC y AUC-PR, además de matriz de confusión y curvas ROC/PR.

In [ ]:
clf_cat = ["corregimiento"]
clf_num = ["anio", "mes", "mes_seno", "mes_coseno", "siniestros_lag_1", "promedio_3_meses_previo"]
clf_features = clf_cat + clf_num
clf_models = {
    "Regresión logística": LogisticRegression(C=0.5, class_weight="balanced", max_iter=2000,
                                                solver="liblinear", random_state=RANDOM_STATE),
    "Árbol de decisión": DecisionTreeClassifier(max_depth=6, min_samples_leaf=8,
                                                 class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(n_estimators=350, max_depth=12, min_samples_leaf=4,
                                              max_features=0.8, class_weight="balanced_subsample",
                                              n_jobs=-1, random_state=RANDOM_STATE),
}

# Baseline de clase mayoritaria, sin DummyClassifier.
train_prevalence = mon_train.alta_siniestralidad.mean()
baseline_proba = np.repeat(train_prevalence, len(mon_test))
baseline_label = (baseline_proba >= 0.5).astype(int)
clf_rows = [{
    "Modelo": "Baseline: clase mayoritaria",
    "Accuracy": accuracy_score(mon_test.alta_siniestralidad, baseline_label),
    "Precision": precision_score(mon_test.alta_siniestralidad, baseline_label, zero_division=0),
    "Recall": recall_score(mon_test.alta_siniestralidad, baseline_label, zero_division=0),
    "F1": f1_score(mon_test.alta_siniestralidad, baseline_label, zero_division=0),
    "AUC-ROC": roc_auc_score(mon_test.alta_siniestralidad, baseline_proba),
    "AUC-PR": average_precision_score(mon_test.alta_siniestralidad, baseline_proba),
}]

clf_fitted = {}
for name, estimator in clf_models.items():
    pipe = Pipeline([("prep", preprocessor(clf_cat, clf_num)), ("model", estimator)])
    pipe.fit(mon_train[clf_features], mon_train.alta_siniestralidad)
    proba = pipe.predict_proba(mon_test[clf_features])[:, 1]
    label = (proba >= 0.5).astype(int)
    clf_rows.append({"Modelo": name, "Accuracy": accuracy_score(mon_test.alta_siniestralidad, label),
                     "Precision": precision_score(mon_test.alta_siniestralidad, label, zero_division=0),
                     "Recall": recall_score(mon_test.alta_siniestralidad, label, zero_division=0),
                     "F1": f1_score(mon_test.alta_siniestralidad, label, zero_division=0),
                     "AUC-ROC": roc_auc_score(mon_test.alta_siniestralidad, proba),
                     "AUC-PR": average_precision_score(mon_test.alta_siniestralidad, proba)})
    clf_fitted[name] = pipe
clf_results = pd.DataFrame(clf_rows).sort_values("AUC-ROC", ascending=False)
display(clf_results.round(4))

# Optimización temporal del clasificador.
clf_val_year = test_year - 1
clf_fit_tune = monthly[monthly.anio < clf_val_year].copy()
clf_val_tune = monthly[monthly.anio == clf_val_year].copy()
clf_fit_tune["alta_siniestralidad"] = (clf_fit_tune.siniestros_mes >= HIGH_THRESHOLD).astype(int)
clf_val_tune["alta_siniestralidad"] = (clf_val_tune.siniestros_mes >= HIGH_THRESHOLD).astype(int)
clf_spaces = {
    "Regresión logística": {"model__C": np.logspace(-2, 1, 15)},
    "Árbol de decisión": {
        "model__max_depth": [3, 4, 5, 6, 8, 10],
        "model__min_samples_leaf": [2, 4, 6, 8, 12, 16],
    },
    "Random Forest": {
        "model__n_estimators": [200, 350, 500],
        "model__max_depth": [6, 8, 12, 16, None],
        "model__min_samples_leaf": [2, 4, 6, 8],
        "model__max_features": [0.5, 0.8, 1.0],
    },
}

tuned_clf = {}
for name in ["Regresión logística", "Árbol de decisión", "Random Forest"]:
    sampled = list(ParameterSampler(clf_spaces[name], n_iter=10, random_state=RANDOM_STATE))
    best_auc, best_params = -np.inf, None
    for params in sampled:
        candidate = Pipeline([("prep", preprocessor(clf_cat, clf_num)), ("model", clf_models[name])])
        candidate.set_params(**params)
        candidate.fit(clf_fit_tune[clf_features], clf_fit_tune.alta_siniestralidad)
        val_proba = candidate.predict_proba(clf_val_tune[clf_features])[:, 1]
        auc = roc_auc_score(clf_val_tune.alta_siniestralidad, val_proba)
        if auc > best_auc:
            best_auc, best_params = auc, params
    final_pipe = Pipeline([("prep", preprocessor(clf_cat, clf_num)), ("model", clf_models[name])])
    final_pipe.set_params(**best_params)
    final_pipe.fit(mon_train[clf_features], mon_train.alta_siniestralidad)
    tuned_clf[name] = final_pipe
    print(f"{name}: mejor validación temporal AUC={best_auc:.4f}; parámetros={best_params}")

optimized_clf_rows = []
for name, pipe in tuned_clf.items():
    proba = pipe.predict_proba(mon_test[clf_features])[:, 1]
    label = (proba >= 0.5).astype(int)
    optimized_clf_rows.append({"Modelo optimizado": name,
        "Accuracy": accuracy_score(mon_test.alta_siniestralidad, label),
        "Precision": precision_score(mon_test.alta_siniestralidad, label, zero_division=0),
        "Recall": recall_score(mon_test.alta_siniestralidad, label, zero_division=0),
        "F1": f1_score(mon_test.alta_siniestralidad, label, zero_division=0),
        "AUC-ROC": roc_auc_score(mon_test.alta_siniestralidad, proba),
        "AUC-PR": average_precision_score(mon_test.alta_siniestralidad, proba)})
optimized_clf_results = pd.DataFrame(optimized_clf_rows).sort_values("AUC-ROC", ascending=False)
display(optimized_clf_results.round(4))

best_clf_name = optimized_clf_results.iloc[0]["Modelo optimizado"]
best_pipe = tuned_clf[best_clf_name]
best_proba = best_pipe.predict_proba(mon_test[clf_features])[:, 1]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.heatmap(confusion_matrix(mon_test.alta_siniestralidad, best_proba >= .5), annot=True, fmt="d", ax=axes[0])
axes[0].set_title(f"Matriz de confusión: {best_clf_name}")
RocCurveDisplay.from_predictions(mon_test.alta_siniestralidad, best_proba, ax=axes[1])
PrecisionRecallDisplay.from_predictions(mon_test.alta_siniestralidad, best_proba, ax=axes[2])
plt.tight_layout()
plt.show()

## 8. IA como revisora crítica: selección del modelo

**Prompt usado:**

> Para clasificar si el próximo mes tendrá alta siniestralidad por corregimiento, definimos el umbral usando solo entrenamiento y comparamos regresión logística, árbol y Random Forest. ¿Qué debo revisar antes de aceptar un AUC alto como evidencia de utilidad operativa?

**Respuesta crítica sintetizada:** un AUC alto no elimina problemas de desbalance, cambio temporal o subregistro. Deben revisarse matriz de confusión, precision, recall, F1, AUC-PR y la estabilidad del resultado en el año de prueba; además, el umbral debe poder explicarse al stakeholder.

**Decisión del equipo:** la selección se hace con validación temporal y múltiples métricas. La alerta se presenta como priorización agregada y no como afirmación causal ni como predicción individual.

## 9. Ética, sesgos y limitaciones

- Los datos reflejan lo que se registra; no equivalen necesariamente a todos los siniestros.
- Una mayor frecuencia registrada puede reflejar mejor reporte o vigilancia, no mayor riesgo real.
- Los corregimientos pueden correlacionarse con desigualdad de infraestructura y vigilancia. El producto debe asignar **apoyo preventivo**, nunca justificar estigmas, sanciones individuales o controles discriminatorios.
- El umbral de alta siniestralidad es una regla operativa basada en la distribución histórica del entrenamiento; no es una frontera natural ni causal.
- Los lags requieren que la entidad actualice los conteos mensuales antes de usar la app.
- La herramienta no descubre causas ni responsabilidad: una inspección técnica y participación local son indispensables.

### IA como revisora crítica — aplicación web

**Prompt usado:**

> Mi aplicación muestra una estimación de siniestros mensuales y una alerta de alta siniestralidad a una Secretaría de Tránsito. ¿Qué necesita el usuario para interpretar ambas salidas y qué usos debería prohibir explícitamente?

**Crítica sintetizada:** una predicción aislada no es suficiente; la app debe explicar el umbral, diferenciar conteo esperado de probabilidad de alerta, mostrar el alcance y dejar claro que el resultado orienta revisión humana y no sanciones automáticas.

**Acción tomada:** la app muestra ambas salidas, explica la definición de alta siniestralidad, conserva advertencias de uso y propone acciones preventivas sujetas a validación en terreno.

## 10. Conclusiones y trabajo futuro

El proyecto conserva el análisis exploratorio original y reorganiza el modelado alrededor de una misma decisión mensual: **cuánto riesgo operativo esperar y si el próximo mes entra en un escenario de alta siniestralidad por corregimiento**. La decisión de usar los modelos depende de la validación temporal y de su comparación con referencias simples, no de una métrica aislada. Próximos pasos: mejorar la estandarización territorial, incorporar exposición vehicular y otros factores externos cuando el curso lo permita, y validar los resultados con la Secretaría y trabajo de campo.